# Data Cleaning — Financial Manager AI

## Objective and Approved Scope

Implement the approved [cleaning plan](04_cleaning_plan.ipynb) and the user's explicit implementation decisions of 2026-09-28. Earlier planning restrictions do not authorize other implementation work; this approval covers only this notebook and the documented run-specific processed artifacts.

Each source row and all original field values are retained. Raw files and notebooks 01–04 are immutable. No imputation, row deletion, outlier modification, FX, business-identity inference, ratios, growth, normalized measures, anomaly scores, ML, database, RAG, or application functionality is implemented.

Receiver General fiscal dimensions remain independent of Accounting Effective Date. Its summary/item relationships and exact redistribution provenance remain unresolved. Company Financials remains a **sales-record dataset**, not a company panel. Both datasets retain currency status **UNKNOWN**. The approved Profit-parentheses convention is dataset-specific, not a claim that all accounting sources use it.

**Publication contract:** every hard gate must pass before any processed data is published. CSV payloads are first serialized and reloaded in memory. Only then are the five artifacts staged in a new hidden directory, checked from disk, and atomically renamed to a unique run directory. Existing runs are never overwritten. A failed gate raises an error and prevents publication; no data repair is used to make a gate pass.

**Raw token meaning:** raw companion columns preserve exact decoded CSV field contents, including whitespace, grouping, signs, and empty tokens. Original quoting and line endings remain byte-for-byte available in the immutable source file identified by its hash.

## 1. Imports, Source Versions, and Validation Gate

Only existing dependencies and Python's standard library are used. Input hashes pin this approved run to the reviewed files. A different source/notebook version requires review instead of silently extending the rules. Decimal arithmetic uses high precision and raises on inexact/rounded arithmetic; floating point is used only for a labeled EDA-comparison diagnostic, never preservation authority.

In [1]:
from pathlib import Path
from datetime import date, datetime, timezone
from decimal import Decimal, getcontext, Inexact, Rounded, InvalidOperation
from collections import Counter, defaultdict
import csv
import io
import json
import hashlib
import re
import os
import tempfile
import shutil
import uuid
import sys
import platform
import pandas as pd
from IPython.display import display

project_root = Path.cwd() if (Path.cwd()/"data/raw").exists() else Path.cwd().parent
RULE_VERSION = "approved-cleaning-2026-09-28-v1"
EXPECTED_HASHES = {'data/raw/Company Financials Dataset/Financials.csv': '86b856a1d4453fc1fe76afbc7dff8ed90aa6f0144a841c8daf5cd8934bca6d51', 'data/raw/Receiver General Accounting Transactions/tcrg-rgat.csv': 'c8c5f31461be56af8081d17174a820a0eb1b70514889a8433df2e1b6ab91879d', 'notebooks/01_dataset_inventory.ipynb': '7990fff489122c7df67b897329d6247cbe86c8a1f3ecbdcffaa7ed6f3f60c1fa', 'notebooks/02_eda_receiver_general.ipynb': 'db5f07ee7ed93fbbaebaca0a8563aa3a83742be311760fea2e4ee91169ac0024', 'notebooks/03_eda_company_financials.ipynb': 'b4cac62700d2ac515e3dfe55b439d25b77df849ff7c514135a950e06d55fd815', 'notebooks/04_cleaning_plan.ipynb': 'cbc62d7f42e05df4501c2871d976ff80e940813592663eaa32db263a495d1103'}
SOURCE_PATHS = {
    "receiver_general": "data/raw/Receiver General Accounting Transactions/tcrg-rgat.csv",
    "company_financials": "data/raw/Company Financials Dataset/Financials.csv",
}
getcontext().prec = 60
for signal in [Inexact, Rounded, InvalidOperation]:
    getcontext().traps[signal] = True
hard_checks = []

def gate(name, condition, detail=None):
    entry = {"check": name, "passed": bool(condition), "detail": detail}
    hard_checks.append(entry)
    if not condition:
        print(json.dumps({"BLOCKER": entry, "publication": "NOT AUTHORIZED BY GATES"}, indent=2, default=str))
        raise RuntimeError(f"Hard gate failed: {name}; no processed publication permitted")

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def verify_protected(label):
    current = {rel: sha256_bytes((project_root/rel).read_bytes()) for rel in EXPECTED_HASHES}
    gate(label, current == EXPECTED_HASHES, current)
    return current

initial_hashes = verify_protected("approved_input_versions")
print("Pinned source/notebook versions verified:", len(initial_hashes))

Pinned source/notebook versions verified: 6


## 2. Explicit Header Maps and Immutable Source Snapshots

The exact original headers map one-to-one to plan-approved names. A source-preserving CSV read retains leading zeros and literal `None`; **only an empty source field** is classified as source missing. No default NA vocabulary is applied. The token DataFrames are immutable snapshots for reconciliation, not working cleaning tables.

In [2]:
RG_MAP = {
    "Accounting-Control-Number-Numéro-contrôle-comptable": "accounting_control_number",
    "Journal-Voucher-Type-Code-Code-du-type-de-pièce-de-journal": "journal_voucher_type_code",
    "Fiscal-Year-Année-Fiscale": "fiscal_year",
    "Journal-Voucher-Identifier-Identificateur-de-la-pièce-de-journal": "journal_voucher_identifier",
    "Accounting-Effective-Date-Date-d'entrée-en-vigueur-comptable": "accounting_effective_date",
    "DepartmentNumber-Numéro-de-Ministère": "department_number",
    "Fiscal-Month-Mois-Fiscal": "fiscal_month",
    "Journal-Voucher-Item-Identifier-Identificateur-de-l'item-de-la-pièce-de-journal": "journal_voucher_item_identifier",
    "Financial-Coding-Block-Bloc-de-codage-financier": "financial_coding_block",
    "Subleger-Account-Identifier-Compte-du-grand-livre-auxiliaire": "subledger_account_identifier",
    "Credit/Debit-Code-Code-Crédit/Débit": "credit_debit_code",
    "Journal-Voucher-Item-Amount-Montant-de-l'item-de-la-pièce-de-journal": "journal_voucher_item_amount",
    "Control-Data-Type-Type-de-données-de-contrôle": "control_data_type",
    "Accounting-Summary-Transaction-Line-Count-Sommaire-comptable-transaction-compte-ligne": "accounting_summary_transaction_line_count",
    "GBS-Control-Number-Numéro-de-contrôle-du-SBG": "gbs_control_number",
    "General-Ledger-Account-Code-Code-du-compte-du-grand-livre-général": "general_ledger_account_code",
}
CF_MAP = {
    "Segment":"segment", "Country":"country", " Product ":"product", " Discount Band ":"discount_band",
    " Units Sold ":"units_sold", " Manufacturing Price ":"manufacturing_price", " Sale Price ":"sale_price",
    " Gross Sales ":"gross_sales", " Discounts ":"discounts", "  Sales ":"sales", " COGS ":"cogs", " Profit ":"profit",
    "Date":"period_date", "Month Number":"month_number", " Month Name ":"month_name", "Year":"year",
}
MAPS = {"receiver_general":RG_MAP, "company_financials":CF_MAP}
RG_IDS = ["accounting_control_number","journal_voucher_identifier","department_number","journal_voucher_item_identifier","subledger_account_identifier","gbs_control_number"]
CF_MEASURES = ["units_sold","manufacturing_price","sale_price","gross_sales","discounts","sales","cogs","profit"]
CF_TRIM = {"product","discount_band","month_name"}
KEYS = {
    "receiver_general":["accounting_control_number","journal_voucher_item_identifier"],
    "company_financials":["segment","country","product","discount_band","period_date","sale_price"],
}

def read_source(dataset):
    source_encoding = "utf-8-sig" if dataset == "receiver_general" else "utf-8"
    with (project_root/SOURCE_PATHS[dataset]).open("r",encoding=source_encoding,newline="") as handle:
        reader = csv.DictReader(handle)
        headers = reader.fieldnames
        rows = list(reader)
    gate(dataset+".headers",headers == list(MAPS[dataset]))
    gate(dataset+".csv_width",all(set(row)==set(headers) and all(isinstance(v,str) for v in row.values()) for row in rows))
    return headers,rows

source_rows = {}
raw_frames = {}
raw_snapshots = {}
for dataset in MAPS:
    headers, rows = read_source(dataset)
    source_rows[dataset] = rows
    raw_frames[dataset] = pd.DataFrame(rows,columns=headers,dtype="string")
    raw_snapshots[dataset] = raw_frames[dataset].copy(deep=True)
    gate(dataset+".mapping_bijection",len(headers)==16 and len(set(MAPS[dataset].values()))==16)
    gate(dataset+".row_count",len(rows)==(9282 if dataset=="receiver_general" else 700))
    gate(dataset+".exact_source_duplicates",len(rows)==len({tuple(r[h] for h in headers) for r in rows}))
expected_rg_missing = [0,26,0,9104,0,26,0,0,9000,2186,0,0,178,26,239,9091]
source_null_counts = {d:{MAPS[d][h]:sum(r[h]=="" for r in source_rows[d]) for h in MAPS[d]} for d in MAPS}
gate("receiver_general.source_nulls",list(source_null_counts["receiver_general"].values())==expected_rg_missing)
gate("receiver_general.source_null_total",sum(source_null_counts["receiver_general"].values())==29876)
gate("company_financials.source_nulls",sum(source_null_counts["company_financials"].values())==0)
display(pd.DataFrame([{"dataset":d,"rows":len(source_rows[d]),"columns":16,"source_missing_cells":sum(source_null_counts[d].values())} for d in MAPS]))

,dataset,rows,columns,source_missing_cells
0,receiver_general,9282,16,29876
1,company_financials,700,16,0


## 3. Source-Validated Numerical Grammar

Patterns are intentionally narrower than a general financial parser. Receiver General item amounts are nonnegative ungrouped tokens with exactly two fractional digits in this extract. Company Financials patterns have literal observed spaces and dollar signs: plain magnitudes, one-comma groups, or Indian-style multi-comma groups. No Western million grouping, exponent, percentages, extra currency symbols, alternate signs, or arbitrary whitespace is accepted.

Parentheses are accepted **only for Profit** and only in the observed plain/thousands layouts. Dollar-dash is accepted **only for Discounts and Profit**, producing null plus `unresolved_placeholder`. Each field's enabled pattern families must occur in the pinned current source. Any unrecognized token is retained, reported, and blocks publication; the whitelist is never broadened automatically.

In [3]:
CF_PATTERNS = {
    "plain": r" \$(?P<magnitude>[0-9]{1,3}\.[0-9]{2}) ",
    "thousands": r" \$(?P<magnitude>[0-9]{1,2},[0-9]{3}\.[0-9]{2}) ",
    "indian": r" \$(?P<magnitude>[0-9]{1,2},[0-9]{2},[0-9]{3}\.[0-9]{2}) ",
    "negative_plain": r" \$\((?P<magnitude>[0-9]{1,3}\.[0-9]{2})\)",
    "negative_thousands": r" \$\((?P<magnitude>[0-9]{1,2},[0-9]{3}\.[0-9]{2})\)",
    "dollar_dash": r" \$-   ",
}
CF_ALLOWED = {
    "units_sold":["plain","thousands"], "manufacturing_price":["plain"], "sale_price":["plain"],
    "gross_sales":["thousands","indian"], "discounts":["plain","thousands","indian","dollar_dash"],
    "sales":["thousands","indian"], "cogs":["plain","thousands","indian"],
    "profit":["plain","thousands","indian","negative_plain","negative_thousands","dollar_dash"],
}
RG_AMOUNT_PATTERN = r"[0-9]+\.[0-9]{2}"

def parse_cf(field, token):
    if token == "": return None,"source_missing","empty source field"
    for family in CF_ALLOWED[field]:
        match = re.fullmatch(CF_PATTERNS[family],token)
        if match:
            if family == "dollar_dash": return None,"unresolved_placeholder",family
            magnitude = Decimal(match["magnitude"].replace(",",""))
            value = -magnitude if family.startswith("negative_") else magnitude
            return value,"parsed",family
    return None,"parsing_failed","unrecognized field-specific format"

format_observations = {}
format_failures = []
for source,field in CF_MAP.items():
    if field not in CF_MEASURES: continue
    families = Counter()
    for row_number,row in enumerate(source_rows["company_financials"],1):
        value,status,reason=parse_cf(field,row[source])
        families[reason]+=1
        if status=="parsing_failed":format_failures.append({"row":row_number,"field":field,"raw":row[source]})
    format_observations[field]=dict(families)
print("Observed Company Financials families by field:")
display(pd.DataFrame(format_observations).fillna(0).astype(int))
# fillna above is display-only count formatting; no source or cleaned value is filled.
print("Unrecognized financial tokens:",format_failures)
gate("company_financials.observed_grammar_only",not format_failures and all(set(format_observations[f])==set(CF_ALLOWED[f]) for f in CF_MEASURES),format_failures)
gate("receiver_general.observed_amount_grammar",all(re.fullmatch(RG_AMOUNT_PATTERN,r[list(RG_MAP)[11]]) for r in source_rows["receiver_general"]))

Observed Company Financials families by field:


,units_sold,manufacturing_price,sale_price,gross_sales,discounts,sales,cogs,profit
thousands,499,0,0,432,477,437,441,562
plain,201,700,700,0,155,0,1,23
indian,0,0,0,268,15,263,258,52
dollar_dash,0,0,0,0,53,0,0,5
negative_thousands,0,0,0,0,0,0,0,56
negative_plain,0,0,0,0,0,0,0,2


Unrecognized financial tokens: []


## 4. Separate Standardized Records and Raw Companions

Every source field has a standardized field, a `<field>_raw` companion, and a `<field>_status`. Identifiers/codes are strings from source tokens; integer temporal/count fields are validated separately. Company Financials outer-space trimming applies only to Product, Discount Band, and Month Name. Literal `None`, case, accents, and internal spacing are preserved.

Lineage uses a one-based **CSV data-record number** (excluding the header), source path, and source SHA-256. It is not a transaction/company identifier. Currency metadata applies only to the monetary fields listed in the manifest; Units Sold is excluded. Both codes/declarations remain null and status UNKNOWN.

In [4]:
def field_kind(dataset,field):
    if dataset=="receiver_general":
        if field=="journal_voucher_item_amount":return "decimal"
        if field in {"fiscal_month","accounting_summary_transaction_line_count"}:return "integer"
        if field=="accounting_effective_date":return "date"
    else:
        if field in CF_MEASURES:return "decimal"
        if field in {"month_number","year"}:return "integer"
        if field=="period_date":return "date"
    return "string"

def parse_field(dataset,field,token):
    if token=="":return None,"source_missing","empty source field"
    kind=field_kind(dataset,field)
    if dataset=="company_financials" and field in CF_MEASURES:return parse_cf(field,token)
    if kind=="decimal":
        if not re.fullmatch(RG_AMOUNT_PATTERN,token):return None,"parsing_failed","unexpected amount grammar"
        return Decimal(token),"parsed","ungrouped_decimal_2_places"
    if kind=="integer":
        if not re.fullmatch(r"[0-9]+",token):return None,"parsing_failed","non-digit integer token"
        return int(token),"parsed","exact integer interpretation"
    if kind=="date":
        pattern,fmt=(r"[0-9]{4}-[0-9]{2}-[0-9]{2}","%Y-%m-%d") if dataset=="receiver_general" else (r"[0-9]{2}/[0-9]{2}/[0-9]{4}","%d/%m/%Y")
        if not re.fullmatch(pattern,token):return None,"parsing_failed","unexpected date format"
        try:value=datetime.strptime(token,fmt).date()
        except ValueError:return None,"parsing_failed","invalid date"
        if value.strftime(fmt)!=token:return None,"parsing_failed","date does not round-trip"
        return value,"parsed",fmt
    if dataset=="company_financials" and field in CF_TRIM:
        return token.strip(" "),"standardized" if token!=token.strip(" ") else "kept","outer ASCII spaces only"
    return token,"kept","exact source token"

lineage_spec = {
    "source_file":{"type":"string","nullable":False},
    "source_sha256":{"type":"string","nullable":False},
    "source_row_number":{"type":"integer","nullable":False},
    "original_currency":{"type":"string","nullable":True},
    "currency_code":{"type":"string","nullable":True},
    "currency_source":{"type":"string","nullable":True},
    "currency_status":{"type":"string","nullable":False},
}
SCHEMAS={}
RULE_IDS={}
for dataset,mapping in MAPS.items():
    schema={};rule_ids={}
    for number,(header,field) in enumerate(mapping.items(),1):
        rule_ids[field]=("RG" if dataset=="receiver_general" else "CF")+f"{number:02d}"
        schema[field]={"type":field_kind(dataset,field),"nullable":True}
        schema[field+"_raw"]={"type":"string","nullable":False}
        schema[field+"_status"]={"type":"string","nullable":False}
    schema.update(lineage_spec)
    SCHEMAS[dataset]=schema;RULE_IDS[dataset]=rule_ids
    gate(dataset+".unique_output_headers",len(schema)==16*3+len(lineage_spec))

def transform(dataset):
    records=[];counts={field:Counter() for field in MAPS[dataset].values()};flags=[]
    for row_number,source_row in enumerate(source_rows[dataset],1):
        record={}
        for source,field in MAPS[dataset].items():
            token=source_row[source]
            value,status,reason=parse_field(dataset,field,token)
            record[field]=value;record[field+"_raw"]=token;record[field+"_status"]=status
            counts[field][status]+=1
            if status in {"source_missing","unresolved_placeholder","parsing_failed"}:
                flags.append({"dataset":dataset,"scope":"field","source_file":SOURCE_PATHS[dataset],"source_sha256":initial_hashes[SOURCE_PATHS[dataset]],"source_row_number":str(row_number),"field":field,"rule_id":RULE_IDS[dataset][field],"flag":status,"status":"unresolved" if status!="source_missing" else "observed","raw_value":token,"reason":reason})
        record.update(source_file=SOURCE_PATHS[dataset],source_sha256=initial_hashes[SOURCE_PATHS[dataset]],source_row_number=row_number,
                      original_currency=None,currency_code=None,currency_source=None,currency_status="UNKNOWN")
        records.append(record)
    return records,{field:dict(count) for field,count in counts.items()},flags

cleaned={};parse_counts={};review_flags=[]
for dataset in MAPS:
    records,counts,flags=transform(dataset)
    cleaned[dataset]=records;parse_counts[dataset]=counts;review_flags.extend(flags)
    failed=[flag for flag in flags if flag["flag"]=="parsing_failed"]
    if failed:display(pd.DataFrame(failed))
    gate(dataset+".no_parsing_failures",not failed,failed)
    print(dataset,"statuses by field")
    display(pd.DataFrame(counts).fillna(0).astype(int).T)
    # Display-only absent count cells are zero; base data nulls are never imputed.
    repeat,repeat_counts,repeat_flags=transform(dataset)
    gate(dataset+".deterministic_transformation",repeat==records and repeat_counts==counts and repeat_flags==flags)
print("No source DataFrame was overwritten. Cleaned records are separate Python objects.")

receiver_general statuses by field


,kept,source_missing,parsed
accounting_control_number,9282,0,0
journal_voucher_type_code,9256,26,0
fiscal_year,9282,0,0
journal_voucher_identifier,178,9104,0
accounting_effective_date,0,0,9282
department_number,9256,26,0
fiscal_month,0,0,9282
journal_voucher_item_identifier,9282,0,0
financial_coding_block,282,9000,0
subledger_account_identifier,7096,2186,0


company_financials statuses by field


,kept,standardized,parsed,unresolved_placeholder
segment,700,0,0,0
country,700,0,0,0
product,0,700,0,0
discount_band,0,700,0,0
units_sold,0,0,700,0
manufacturing_price,0,0,700,0
sale_price,0,0,700,0
gross_sales,0,0,700,0
discounts,0,0,647,53
sales,0,0,700,0


No source DataFrame was overwritten. Cleaned records are separate Python objects.


## 5. Identifier, Category, Date, Key, and Null Gates

Check every token/row, representation type, key, date, and collision before serialization. Raw source fields remain independently accessible even where an integer period/count representation omits lexical zero padding. Fiscal labels are not reconciled to calendar months. New collisions are blockers rather than grounds for deduplication.

In [5]:
def source_key(dataset,row,fields):
    reverse={field:header for header,field in MAPS[dataset].items()}
    return tuple(row[reverse[f]] for f in fields)

def repetitions(keys):return len(keys)-len(set(keys))

def category_collisions(dataset,field):
    groups=defaultdict(set)
    for row in cleaned[dataset]:groups[row[field]].add(row[field+"_raw"])
    return {str(value):sorted(tokens) for value,tokens in groups.items() if len(tokens)>1}

key_results={};null_results={};collision_results={}
months=["January","February","March","April","May","June","July","August","September","October","November","December"]
for dataset,records in cleaned.items():
    mapping=MAPS[dataset]
    gate(dataset+".row_lineage",len(records)==len(source_rows[dataset]) and [r["source_row_number"] for r in records]==list(range(1,len(records)+1)))
    gate(dataset+".raw_companions",all(r[f+"_raw"]==source_rows[dataset][i][h] for i,r in enumerate(records) for h,f in mapping.items()))
    gate(dataset+".source_lineage",all(r["source_file"]==SOURCE_PATHS[dataset] and r["source_sha256"]==initial_hashes[SOURCE_PATHS[dataset]] for r in records))
    gate(dataset+".exact_output_schema",all(list(r)==list(SCHEMAS[dataset]) for r in records))
    nullable_counts={f:sum(r[f] is None for r in records) for f in mapping.values()}
    expected_nulls=dict(source_null_counts[dataset])
    if dataset=="company_financials":expected_nulls.update(discounts=53,profit=5)
    gate(dataset+".typed_null_reconciliation",nullable_counts==expected_nulls,{"source":source_null_counts[dataset],"typed":nullable_counts})
    null_results[dataset]={"source":source_null_counts[dataset],"typed":nullable_counts}
    for f in mapping.values():
        counts=parse_counts[dataset][f]
        gate(dataset+"."+f+".status_count",sum(counts.values())==len(records) and counts.get("source_missing",0)==source_null_counts[dataset][f])
    keys_before=[source_key(dataset,r,KEYS[dataset]) for r in source_rows[dataset]]
    keys_after=[tuple(r[f] for f in KEYS[dataset]) for r in records]
    missing_key=sum(any(r[f] is None for f in KEYS[dataset]) for r in records)
    key_results[dataset]={"source_repetitions":repetitions(keys_before),"cleaned_repetitions":repetitions(keys_after),"missing_components":missing_key}
    gate(dataset+".candidate_key",key_results[dataset]=={"source_repetitions":0,"cleaned_repetitions":0,"missing_components":0})
    represented=[tuple(r[f] for f in mapping.values()) for r in records]
    gate(dataset+".represented_full_duplicates",repetitions(represented)==0)
    types_ok=True
    for r in records:
        for field,spec in SCHEMAS[dataset].items():
            value=r[field]
            if value is None:types_ok &= spec["nullable"]
            else:types_ok &= type(value) is {"string":str,"integer":int,"decimal":Decimal,"date":date}[spec["type"]]
    gate(dataset+".semantic_types",types_ok)
    gate(dataset+".currency_unknown_no_fx",all(r["currency_status"]=="UNKNOWN" and all(r[f] is None for f in ["original_currency","currency_code","currency_source"]) for r in records))

rg=cleaned["receiver_general"];cf=cleaned["company_financials"]
gate("receiver_general.identifier_fidelity",all(r[f]==r[f+"_raw"] for r in rg for f in RG_IDS if r[f] is not None))
gate("receiver_general.string_code_fidelity",all(r[f]==(None if r[f+"_raw"]=="" else r[f+"_raw"]) for r in rg for f in RG_MAP.values() if field_kind("receiver_general",f)=="string"))
gate("receiver_general.control_repetition",repetitions([r["accounting_control_number"] for r in rg])==148)
gate("receiver_general.fiscal_dimensions_unchanged",all(r["fiscal_year"]==r["fiscal_year_raw"] and re.fullmatch(r"[0-9]{4}/[0-9]{4}",r["fiscal_year"]) and r["fiscal_month"]==int(r["fiscal_month_raw"]) and 1<=r["fiscal_month"]<=12 for r in rg))
gate("receiver_general.effective_date_roundtrip",all(r["accounting_effective_date"].isoformat()==r["accounting_effective_date_raw"] for r in rg))
rg_dates=[r["accounting_effective_date"] for r in rg]
gate("receiver_general.date_coverage",min(rg_dates)==date(2023,4,3) and max(rg_dates)==date(2024,1,3) and len(set(rg_dates))==188)
gate("receiver_general.direction_labels",set(r["credit_debit_code"] for r in rg)=={"DR","CR"})
gate("receiver_general.summary_counts_exact",all(r["accounting_summary_transaction_line_count"] is None or Decimal(r["accounting_summary_transaction_line_count_raw"])==r["accounting_summary_transaction_line_count"] for r in rg))
for field in CF_TRIM:
    collisions=category_collisions("company_financials",field)
    collision_results[field]=collisions
    gate("company_financials."+field+".no_trim_collisions",not collisions,collisions)
    gate("company_financials."+field+".trim_only",all(r[field]==r[field+"_raw"].strip(" ") for r in cf))
gate("company_financials.literal_None",sum(r["discount_band"]=="None" for r in cf)==53 and all(r["discount_band"] is not None for r in cf))
gate("company_financials.unmodified_categories",all(r[f]==r[f+"_raw"] for r in cf for f in ["segment","country"]))
gate("company_financials.temporal_labels",all(r["period_date"].strftime("%d/%m/%Y")==r["period_date_raw"] and r["period_date"].day==1 and r["period_date"].month==r["month_number"] and r["period_date"].year==r["year"] and months[r["month_number"]-1]==r["month_name"] for r in cf))
cf_periods=sorted({(r["year"],r["month_number"]) for r in cf})
gate("company_financials.period_coverage",cf_periods==[(2013,m) for m in range(9,13)]+[(2014,m) for m in range(1,13)])
cf_five=KEYS["company_financials"][:-1]
gate("company_financials.five_field_repetitions",repetitions([source_key("company_financials",r,cf_five) for r in source_rows["company_financials"]])==12 and repetitions([tuple(r[f] for f in cf_five) for r in cf])==12)
print("Identifier/category/date/null/key gates passed; no collisions introduced.")
display(pd.DataFrame(key_results).T)

Identifier/category/date/null/key gates passed; no collisions introduced.


,source_repetitions,cleaned_repetitions,missing_components
receiver_general,0,0,0
company_financials,0,0,0


## 6. Exact Numeric Preservation and Reconciliation

An independent integer-digit reference parses accepted source magnitudes without the production Decimal string parser. Decimal tuples, including sign and scale, must agree per cell. Exact sums are numerical preservation checks on identical available populations—not currency-comparable financial totals. Per-row preservation also protects every extreme, fractional quantity, negative value, and zero; no statistical threshold changes a value.

Prices are checked per value, not interpreted as additive business totals. The labeled float diagnostic shows how exact decimal sums differ from EDA-style binary arithmetic; it cannot authorize drift in any stored amount.

In [6]:
def reference_from_digits(token,dataset,field):
    if token=="":return None
    if dataset=="company_financials":
        stripped=token.strip(" ")
        if stripped=="$-":return None
        body=stripped[1:]
        negative=body.startswith("(")
        if negative:body=body[1:-1]
    else:body=token;negative=False
    whole,fraction=body.split(".")
    digit_text="".join(whole.split(","))+fraction
    return Decimal((1 if negative else 0,tuple(int(digit) for digit in digit_text),-len(fraction)))

def exact_sum(values):return sum((v for v in values if v is not None),Decimal(0))

numeric_reconciliation={};float_diagnostics={};reference_values={}
for dataset in MAPS:
    numeric_reconciliation[dataset]={};reference_values[dataset]={};float_diagnostics[dataset]={}
    for source,field in MAPS[dataset].items():
        if field_kind(dataset,field)!="decimal":continue
        reference=[reference_from_digits(row[source],dataset,field) for row in source_rows[dataset]]
        actual=[row[field] for row in cleaned[dataset]]
        reference_values[dataset][field]=reference
        exact=all((a is None and b is None) or (a is not None and b is not None and a.as_tuple()==b.as_tuple()) for a,b in zip(actual,reference))
        gate(dataset+"."+field+".exact_token_values",exact)
        available=[x for x in actual if x is not None]
        reference_available=[x for x in reference if x is not None]
        numeric_reconciliation[dataset][field]={"numeric_count":len(available),"null_count":len(actual)-len(available),"minimum":str(min(available)),"maximum":str(max(available)),"negative_count":sum(x<0 for x in available),"zero_count":sum(x==0 for x in available),"all_source_values_preserved":exact}
        if field not in {"sale_price","manufacturing_price"}:
            total=exact_sum(actual);ref_total=exact_sum(reference)
            gate(dataset+"."+field+".exact_total",total==ref_total)
            numeric_reconciliation[dataset][field].update(total=str(total),reference_total=str(ref_total))
            float_total=pd.Series([float(v) for v in available],dtype="float64").sum()
            # Diagnostic only; never used to construct or accept cleaned values.
            float_diagnostics[dataset][field]={"eda_style_float_sum":repr(float(float_total)),"exact_decimal_sum":str(total),"difference_from_shortest_float_text":str(Decimal(str(float_total))-total),"authority":"source decimals only"}
    print(dataset);display(pd.DataFrame(numeric_reconciliation[dataset]).T)

cf_success=sum(parse_counts["company_financials"][f].get("parsed",0) for f in CF_MEASURES)
cf_unresolved=sum(parse_counts["company_financials"][f].get("unresolved_placeholder",0) for f in CF_MEASURES)
gate("company_financials.measure_parse_counts",cf_success==5542 and cf_unresolved==58)
gate("company_financials.placeholder_status",all((r[f] is None and r[f+"_status"]=="unresolved_placeholder") if r[f+"_raw"].strip(" ")=="$-" else r[f+"_status"]=="parsed" for r in cf for f in ["discounts","profit"]))
gate("company_financials.negative_profit_preserved",sum(r["profit"] is not None and r["profit"]<0 for r in cf)==58)
gate("receiver_general.zero_amounts_preserved",sum(r["journal_voucher_item_amount"]==0 for r in rg)==2)
rg_subtotals={}
for dimension in ["credit_debit_code","journal_voucher_type_code","department_number"]:
    actual=defaultdict(lambda:Decimal(0));reference=defaultdict(lambda:Decimal(0))
    for i,r in enumerate(rg):
        actual[r[dimension]]+=r["journal_voucher_item_amount"]
        reference[r[dimension]]+=reference_values["receiver_general"]["journal_voucher_item_amount"][i]
    gate("receiver_general."+dimension+".subtotals",dict(actual)==dict(reference) and exact_sum(actual.values())==exact_sum(reference_values["receiver_general"]["journal_voucher_item_amount"]))
    rg_subtotals[dimension]=[{"category":key,"total":str(value)} for key,value in actual.items()]
print("All numeric values, including extrema, match source-token interpretations exactly.")

receiver_general


,numeric_count,null_count,minimum,maximum,negative_count,zero_count,all_source_values_preserved,total,reference_total
journal_voucher_item_amount,9282,0,0.00,39555794411.27,0,2,True,3061288992129.16,3061288992129.16


company_financials


,numeric_count,null_count,minimum,maximum,negative_count,zero_count,all_source_values_preserved,total,reference_total
units_sold,700,0,200.00,4492.50,0,0,True,1125806.00,1125806.00
manufacturing_price,700,0,3.00,260.00,0,0,True,NaN,NaN
sale_price,700,0,7.00,350.00,0,0,True,NaN,NaN
gross_sales,700,0,1799.00,1207500.00,0,0,True,127931598.50,127931598.50
discounts,647,53,18.41,149677.50,0,0,True,9205248.27,9205248.27
sales,700,0,1655.08,1159200.00,0,0,True,118726350.29,118726350.29
cogs,700,0,918.00,950625.00,0,0,True,101832648.00,101832648.00
profit,695,5,-40617.50,262200.00,58,0,True,16893702.29,16893702.29


All numeric values, including extrema, match source-token interpretations exactly.


## 7. Soft Financial Checks and Review Flags

The three approved Company Financials checks use exact decimals and the plan's 0.011 source-unit review tolerance. They do not repair amounts or fill placeholders. A check with an unavailable operand is untestable, not a failure. Review flags distinguish field-level null/placeholder conditions from dataset-level unresolved semantics/currency.

Optional statistical extreme flags and ratios are not recomputed or exported: their EDA thresholds are not validated error definitions, and every value is already protected by per-row gates. No additional production features are introduced.

In [7]:
soft_checks=[]
soft_definitions=[
    ("units_times_price_vs_gross",["units_sold","sale_price","gross_sales"],lambda r:r["units_sold"]*r["sale_price"]-r["gross_sales"]),
    ("gross_minus_discounts_vs_sales",["gross_sales","discounts","sales"],lambda r:r["gross_sales"]-r["discounts"]-r["sales"]),
    ("sales_minus_cogs_vs_profit",["sales","cogs","profit"],lambda r:r["sales"]-r["cogs"]-r["profit"]),
]
for name,fields,calculation in soft_definitions:
    residuals=[];unavailable=0;outside=[]
    for r in cf:
        if any(r[f] is None for f in fields):unavailable+=1;continue
        residual=calculation(r);residuals.append(residual)
        if abs(residual)>Decimal("0.011"):
            outside.append(r["source_row_number"])
            review_flags.append({"dataset":"company_financials","scope":"record_check","source_file":r["source_file"],"source_sha256":r["source_sha256"],"source_row_number":str(r["source_row_number"]),"field":",".join(fields),"rule_id":"CF-SOFT-"+name,"flag":"financial_relationship_review","status":"review","raw_value":"","reason":"exact residual="+str(residual)})
    soft_checks.append({"check":name,"testable":len(residuals),"untestable":unavailable,"outside_tolerance":len(outside),"outside_rows":outside,"maximum_absolute_residual":str(max(abs(v) for v in residuals)),"tolerance":"0.011","interpretation":"soft consistency only; no corrections"})
gate("company_financials.soft_check_testability",[r["testable"] for r in soft_checks]==[700,647,695])
# A newly discovered material residual requires review, not a new corrective rule.
gate("company_financials.no_new_material_financial_issue",all(r["outside_tolerance"]==0 for r in soft_checks),soft_checks)
for dataset in MAPS:
    descriptions=[("currency_unknown","currency","No authoritative denomination established; UNKNOWN, no code or conversion."),
                  ("source_semantics_unresolved","record_granularity","Exact underlying record grain and business semantics remain unresolved."),
                  ("source_semantics_unresolved","provenance","Complete local-extract/source provenance remains unresolved.")]
    if dataset=="receiver_general":descriptions.append(("fiscal_alignment_unresolved","fiscal_period","Fiscal fields retained independently; no fiscal/calendar inference."))
    else:
        descriptions.extend([("source_semantics_unresolved","entity_scope","Sales records have no company identity; no entity inferred."),
                             ("source_semantics_unresolved","units_sold","Dollar-formatted fractional quantity; no monetary currency assigned."),
                             ("source_semantics_unresolved","profit","Parenthesis sign convention approved for this dataset; original token retained.")])
    for flag,field,reason in descriptions:
        review_flags.append({"dataset":dataset,"scope":"dataset","source_file":SOURCE_PATHS[dataset],"source_sha256":initial_hashes[SOURCE_PATHS[dataset]],"source_row_number":"","field":field,"rule_id":"PLAN-SEMANTICS","flag":flag,"status":"unresolved","raw_value":"","reason":reason})
flag_counts=dict(Counter(f["flag"] for f in review_flags))
gate("review_flag_null_placeholder_reconciliation",flag_counts.get("source_missing")==29876 and flag_counts.get("unresolved_placeholder")==58 and flag_counts.get("parsing_failed",0)==0)
display(pd.DataFrame(soft_checks));display(pd.Series(flag_counts,name="flag_count").to_frame())

,check,testable,untestable,outside_tolerance,outside_rows,maximum_absolute_residual,tolerance,interpretation
0,units_times_price_vs_gross,700,0,0,[],0.0000,0.011,soft consistency only; no corrections
1,gross_minus_discounts_vs_sales,647,53,0,[],0.01,0.011,soft consistency only; no corrections
2,sales_minus_cogs_vs_profit,695,5,0,[],0.00,0.011,soft consistency only; no corrections


,flag_count
source_missing,29876
unresolved_placeholder,58
currency_unknown,2
source_semantics_unresolved,7
fiscal_alignment_unresolved,1


## 8. CSV Type/Null Contract and In-Memory Round Trip

CSV is UTF-8 with all fields quoted and explicit LF output line endings. Decimal text uses fixed-point formatting without float conversion; date values use ISO YYYY-MM-DD. A blank standardized nullable field means null; its raw companion and status distinguish source absence from unresolved placeholders. Raw companion blanks remain **empty strings**, not nulls. The literal category `None` remains text.

Readers must use the manifest type/null contract, not spreadsheet inference or Pandas' default NA/type inference. Below, a standard-library CSV reader reloads every value according to that contract. Decimal sign/scale tuples, string identifiers, raw tokens, dates, integer fields, and statuses must round-trip exactly before publication is eligible.

In [8]:
def serialize_value(value,spec):
    if value is None:
        if not spec["nullable"]:raise ValueError("null in nonnullable field")
        return ""
    kind=spec["type"]
    if kind=="decimal":return format(value,"f")
    if kind=="date":return value.isoformat()
    if kind=="integer":return str(value)
    if kind=="string":return value
    raise ValueError("unknown serialization type")

def serialize_records(records,schema):
    buffer=io.StringIO(newline="")
    writer=csv.DictWriter(buffer,fieldnames=list(schema),quoting=csv.QUOTE_ALL,lineterminator="\n")
    writer.writeheader()
    for row in records:writer.writerow({field:serialize_value(row[field],spec) for field,spec in schema.items()})
    return buffer.getvalue().encode("utf-8")

def reload_records(payload,schema):
    reader=csv.DictReader(io.StringIO(payload.decode("utf-8"),newline=""))
    if reader.fieldnames!=list(schema):raise ValueError("serialized header mismatch")
    result=[]
    for tokens in reader:
        if set(tokens)!=set(schema) or any(v is None for v in tokens.values()):raise ValueError("invalid CSV row width")
        record={}
        for field,spec in schema.items():
            token=tokens[field]
            if token=="" and spec["nullable"]:value=None
            elif spec["type"]=="string":value=token
            elif spec["type"]=="decimal":
                if not re.fullmatch(r"-?[0-9]+\.[0-9]+",token):raise ValueError("invalid decimal output")
                value=Decimal(token)
            elif spec["type"]=="integer":
                if not re.fullmatch(r"[0-9]+",token):raise ValueError("invalid integer output")
                value=int(token)
            elif spec["type"]=="date":value=date.fromisoformat(token)
            else:raise ValueError("invalid manifest type")
            record[field]=value
        result.append(record)
    return result

def typed_identical(left,right,schema):
    if len(left)!=len(right):return False
    for a,b in zip(left,right):
        if list(a)!=list(b):return False
        for field,spec in schema.items():
            x,y=a[field],b[field]
            if type(x) is not type(y):return False
            if x is None:
                if y is not None:return False
            elif isinstance(x,Decimal):
                if x.as_tuple()!=y.as_tuple():return False
            elif x!=y:return False
    return True

CSV_FILENAMES={"receiver_general":"receiver_general_standardized.csv","company_financials":"company_financials_standardized.csv"}
payloads={}
for dataset,records in cleaned.items():
    payload=serialize_records(records,SCHEMAS[dataset])
    loaded=reload_records(payload,SCHEMAS[dataset])
    gate(dataset+".memory_roundtrip",typed_identical(records,loaded,SCHEMAS[dataset]))
    gate(dataset+".deterministic_serialization",payload==serialize_records(records,SCHEMAS[dataset]))
    payloads[CSV_FILENAMES[dataset]]=payload
FLAG_SCHEMA={field:{"type":"string","nullable":False} for field in ["dataset","scope","source_file","source_sha256","source_row_number","field","rule_id","flag","status","raw_value","reason"]}
flags_payload=serialize_records(review_flags,FLAG_SCHEMA)
gate("flags.memory_roundtrip",typed_identical(review_flags,reload_records(flags_payload,FLAG_SCHEMA),FLAG_SCHEMA))
payloads["cleaning_review_flags.csv"]=flags_payload
print("In-memory lossless CSV round trips passed for both datasets and review flags.")

In-memory lossless CSV round trips passed for both datasets and review flags.


## 9. Final Pre-Publication Gate and Run Manifest

All original source fields and their raw/status companions are accounted for by the explicit schema. Only approved source facts and audit metadata are present in base datasets. Inputs and original token DataFrames are rechecked. Publication is authorized only if all hard gates pass; expected unresolved semantics are declared in the manifest.

The manifest records field-level parsers/types, source maps/hashes, currency scope, remaining limitations, artifact hashes, and rules. Validation report and manifest hashes are reported in notebook output; neither embeds its own hash, avoiding circular hash dependencies.

In [9]:
def verify_source_frames(label):
    for dataset in MAPS:
        headers,rows=read_source(dataset)
        fresh=pd.DataFrame(rows,columns=headers,dtype="string")
        try:
            pd.testing.assert_frame_equal(raw_frames[dataset],raw_snapshots[dataset])
            pd.testing.assert_frame_equal(raw_frames[dataset],fresh)
            unchanged=rows==source_rows[dataset]
        except AssertionError:unchanged=False
        gate(dataset+"."+label+".raw_dataframe_unchanged",unchanged)

verify_source_frames("prepublication")
verify_protected("protected_files_before_publication")
gate("all_hard_gates_before_publication",all(check["passed"] for check in hard_checks))
run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")+"_"+uuid.uuid4().hex[:8]
limitations=[
    "Receiver General fiscal/effective-date alignment remains unresolved; source fiscal dimensions are unchanged.",
    "Receiver General summary/item relationships and potential overlap remain unresolved.",
    "Exact local extract provenance is incomplete; Receiver General official source is PSPC with recorded Kaggle redistribution; Company Financials origin remains undocumented.",
    "Company Financials is sales-record data without company identity or company-panel meaning.",
    "Discounts and Profit dollar-dashes remain unresolved, not inferred zeros.",
    "Profit parentheses use only the explicitly approved Company Financials sign convention.",
    "Currency UNKNOWN for both datasets; no currency code, inference, FX, or monetary assignment to Units Sold.",
    "Numerical reconciliation totals are preservation checks, not verified comparable business totals.",
]
manifest={
    "run_id":run_id,"rule_version":RULE_VERSION,"approval":"User implementation approval 2026-09-28, decisions 1–26",
    "plan":"notebooks/04_cleaning_plan.ipynb","plan_sha256":EXPECTED_HASHES["notebooks/04_cleaning_plan.ipynb"],
    "protected_inputs":EXPECTED_HASHES,"environment":{"python":platform.python_version(),"pandas":pd.__version__,"decimal_precision":getcontext().prec,"decimal_traps":["Inexact","Rounded","InvalidOperation"]},
    "csv_contract":{"encoding":"utf-8","quoting":"all fields","line_ending":"LF","null_token":"","null_rule":"empty means null only for nullable manifest fields; raw and status fields are nonnullable strings","decimal":"exact fixed-point text, preserve scale/sign; never load through binary float","date":"ISO YYYY-MM-DD date only","reader":"stdlib csv.DictReader plus manifest-specific type conversion; no default NA inference"},
    "source_token_definition":"Decoded CSV field contents; immutable original bytes preserve original quoting and line endings.",
    "lineage":"source_file + source_sha256 + one-based CSV data-record source_row_number, excluding header; not a business key",
    "rules":{"RG_amount_regex":RG_AMOUNT_PATTERN,"CF_patterns":CF_PATTERNS,"CF_enabled_families":CF_ALLOWED,"CF_observed_family_counts":format_observations,
             "CF_trim_only":sorted(CF_TRIM),"parentheses":"negative only for Profit under approved observed patterns","placeholder":"Discounts/Profit exact dollar-dash -> null + unresolved_placeholder","missing_source":"empty token only","identifiers":"original tokens as strings; no padding/reconstruction","rounding":"none","row_deletion":"none","imputation":"none","outlier_modification":"none","FX":"none","analytical_or_ML_features":"none"},
    "datasets":{},"review_flags_schema":FLAG_SCHEMA,"remaining_limitations":limitations,
    "optional_review_methods_not_run":["statistical extreme thresholds","ratio creation","growth","normalization","record-type inference"],
}
for dataset in MAPS:
    monetary=["journal_voucher_item_amount"] if dataset=="receiver_general" else [f for f in CF_MEASURES if f!="units_sold"]
    manifest["datasets"][dataset]={"source_file":SOURCE_PATHS[dataset],"source_sha256":EXPECTED_HASHES[SOURCE_PATHS[dataset]],"rows":len(cleaned[dataset]),"original_header_map":MAPS[dataset],"field_rule_ids":RULE_IDS[dataset],"output_file":CSV_FILENAMES[dataset],"output_schema":SCHEMAS[dataset],"candidate_key":KEYS[dataset],"currency":{"status":"UNKNOWN","code":None,"source":None,"monetary_fields":monetary,"date_field":"accounting_effective_date" if dataset=="receiver_general" else "period_date","scope":"monetary metadata only; no inferred shared denomination","quantity_excluded":["units_sold"] if dataset=="company_financials" else []}}
print("Pre-publication gates passed:",len(hard_checks))
print("Run ID:",run_id)
print("Planned artifacts:",list(payloads)+["cleaning_manifest.json","cleaning_validation_report.json"])

Pre-publication gates passed: 130
Run ID: 20260928T190320706702Z_0a18cf5b
Planned artifacts: ['receiver_general_standardized.csv', 'company_financials_standardized.csv', 'cleaning_review_flags.csv', 'cleaning_manifest.json', 'cleaning_validation_report.json']


## 10. Stage, Reload, and Publish the Approved Artifacts

The staged files are not published datasets. Both dataset CSVs and the flags report are verified from disk before the directory is renamed to the run ID. A second disk read validates the final published paths. Any exception removes only this run's generated staging/output directory, retains the raw sources, and reports that no processed publication remains. No existing run is replaced.

In [10]:
processed_root=project_root/"data/processed"
final_dir=processed_root/run_id
stage_dir=None
published_by_this_cell=False
output_hashes={}

def json_bytes(value):return (json.dumps(value,ensure_ascii=False,indent=2,sort_keys=False)+"\n").encode("utf-8")

def validate_payload_directory(directory,phase,record_gates=True):
    results={}
    disk_manifest = json.loads((directory/"cleaning_manifest.json").read_text(encoding="utf-8")) if (directory/"cleaning_manifest.json").exists() else manifest
    for dataset in MAPS:
        path=directory/CSV_FILENAMES[dataset]
        binary=path.read_bytes()
        ok=binary==payloads[CSV_FILENAMES[dataset]] and typed_identical(cleaned[dataset],reload_records(binary,disk_manifest["datasets"][dataset]["output_schema"]),SCHEMAS[dataset])
        results[dataset]=ok
        if record_gates:gate(dataset+"."+phase+".disk_roundtrip",ok)
        elif not ok:raise RuntimeError(dataset+" final disk verification failed")
    binary=(directory/"cleaning_review_flags.csv").read_bytes()
    ok=binary==flags_payload and typed_identical(review_flags,reload_records(binary,FLAG_SCHEMA),FLAG_SCHEMA)
    results["review_flags"]=ok
    if record_gates:gate(phase+".flags_disk_roundtrip",ok)
    elif not ok:raise RuntimeError("Final flags disk verification failed")
    return results

gate("publication_gate_still_passes",all(c["passed"] for c in hard_checks))
try:
    gate("new_run_directory",not final_dir.exists())
    processed_root.mkdir(parents=True,exist_ok=True)
    stage_dir=Path(tempfile.mkdtemp(prefix=".pending_"+run_id+"_",dir=processed_root))
    for filename,binary in payloads.items():
        with (stage_dir/filename).open("xb") as handle:handle.write(binary)
    stage_results=validate_payload_directory(stage_dir,"staged")
    verify_source_frames("staged")
    verify_protected("protected_files_after_staging")
    gate("all_checks_before_atomic_publish",all(c["passed"] for c in hard_checks))
    report={"run_id":run_id,"rule_version":RULE_VERSION,"result":"PASS","hard_checks":hard_checks,
            "source_null_counts":source_null_counts,"null_reconciliation":null_results,"parse_counts":parse_counts,
            "key_results":key_results,"category_collisions":collision_results,"numeric_reconciliation":numeric_reconciliation,
            "receiver_general_subtotals":rg_subtotals,"float_comparison_diagnostic_not_authority":float_diagnostics,
            "soft_financial_checks":soft_checks,"review_flag_counts":flag_counts,"staged_disk_roundtrip":stage_results,
            "postpublication_verification":"Executed immediately after rename; final result recorded in notebook output. Any failure retracts only this generated run.",
            "scope":{"rows_removed":0,"rows_added":0,"imputations":0,"outliers_modified":0,"FX_conversions":0,"models_trained":0,"analytical_features_in_base_data":0},
            "remaining_limitations":limitations}
    report_binary=json_bytes(report)
    with (stage_dir/"cleaning_validation_report.json").open("xb") as handle:handle.write(report_binary)
    artifact_hashes={filename:sha256_bytes(binary) for filename,binary in payloads.items()}
    artifact_hashes["cleaning_validation_report.json"]=sha256_bytes(report_binary)
    manifest["artifact_sha256"]=artifact_hashes
    manifest["manifest_self_hash"]="Printed in notebook output; omitted here to avoid a circular hash."
    manifest_binary=json_bytes(manifest)
    with (stage_dir/"cleaning_manifest.json").open("xb") as handle:handle.write(manifest_binary)
    loaded_manifest=json.loads((stage_dir/"cleaning_manifest.json").read_text(encoding="utf-8"))
    loaded_report=json.loads((stage_dir/"cleaning_validation_report.json").read_text(encoding="utf-8"))
    if loaded_manifest!=manifest or loaded_report!=report:raise RuntimeError("Metadata round-trip failure")
    for filename,digest in artifact_hashes.items():
        if sha256_bytes((stage_dir/filename).read_bytes())!=digest:raise RuntimeError("Staged artifact hash mismatch")
    if final_dir.exists():raise RuntimeError("Refusing to overwrite an existing run")
    stage_dir.rename(final_dir)
    published_by_this_cell=True
    final_results=validate_payload_directory(final_dir,"published",record_gates=False)
    output_hashes={p.name:sha256_bytes(p.read_bytes()) for p in sorted(final_dir.iterdir()) if p.is_file()}
    if output_hashes!={**artifact_hashes,"cleaning_manifest.json":sha256_bytes(manifest_binary)}:raise RuntimeError("Published output hash mismatch")
    if json.loads((final_dir/"cleaning_manifest.json").read_text())!=manifest:raise RuntimeError("Published manifest mismatch")
    if json.loads((final_dir/"cleaning_validation_report.json").read_text())!=report:raise RuntimeError("Published report mismatch")
    for rel,digest in EXPECTED_HASHES.items():
        if sha256_bytes((project_root/rel).read_bytes())!=digest:raise RuntimeError("Protected file changed after publication")
    for dataset in MAPS:
        pd.testing.assert_frame_equal(raw_frames[dataset],raw_snapshots[dataset])
except Exception:
    # These paths were created by this cell only; immutable source paths are never removed.
    if published_by_this_cell and final_dir.exists():shutil.rmtree(final_dir)
    elif stage_dir is not None and stage_dir.exists():shutil.rmtree(stage_dir)
    print("BLOCKER: generated run retracted/not published; raw files were not changed by publication.")
    raise
print("PASS: all hard gates, staged disk round trips, final disk reloads, metadata and output hashes verified.")
print("Published run:",final_dir.relative_to(project_root))
display(pd.DataFrame([{"file":filename,"sha256":digest} for filename,digest in output_hashes.items()]))

PASS: all hard gates, staged disk round trips, final disk reloads, metadata and output hashes verified.
Published run: data/processed/20260928T190320706702Z_0a18cf5b


,file,sha256
0,cleaning_manifest.json,720165d12ecf27e87823376621c0ae7b0b20fca91130ae...
1,cleaning_review_flags.csv,51a4bbe758c9fa7a014b2f563e3c1bd9c8baa182361077...
2,cleaning_validation_report.json,1c9bffbf5903b27faa68a5059af42016ed01df09f6282b...
3,company_financials_standardized.csv,5245acda56b7d0cb8cc547c913c5abfcde20a55c4b35fb...
4,receiver_general_standardized.csv,67b2605a6e8576fb3f41aedb1bcb053db5ea1ddb33ed69...


## 11. Completion and Remaining Limits

The saved standardized files preserve all input rows, source field contents, identifier zeros/alphanumerics, decimal amounts, and null/placeholder distinctions. Only documented parsing, explicit header mappings, and three Company Financials outer-whitespace standardizations are applied. Every source field is recoverable through its raw companion; all rows retain source hash/path/record position.

The outputs do not resolve fiscal alignment, summary/item overlap, source provenance gaps, entity identity, currency denomination, or the true meaning of dollar-dashes. The approved Profit parsing convention is recorded explicitly. No currency is inferred, no base currency is selected, and no FX or model/application/schema work follows this run.

In [11]:
summary = {
    "run_id":run_id,"receiver_general_rows":len(rg),"company_financials_rows":len(cf),
    "receiver_general_amount_parses":parse_counts["receiver_general"]["journal_voucher_item_amount"].get("parsed",0),
    "company_financials_numeric_parses":cf_success,"company_financials_unresolved_placeholders":cf_unresolved,
    "parsing_failures":sum(counts.get("parsing_failed",0) for dataset in parse_counts.values() for counts in dataset.values()),
    "source_missing_cells_receiver_general":sum(source_null_counts["receiver_general"].values()),
    "hard_checks_passed":len(hard_checks),"published_reload_passed":all(final_results.values()),
    "row_deletions":0,"imputations":0,"outlier_modifications":0,"FX_conversions":0,
    "processed_files":list(output_hashes),"output_hashes":output_hashes,
}
print(json.dumps(summary,indent=2))
print("STOP: no database, FX, ML, RAG, or application implementation is authorized by this notebook.")

{
  "run_id": "20260928T190320706702Z_0a18cf5b",
  "receiver_general_rows": 9282,
  "company_financials_rows": 700,
  "receiver_general_amount_parses": 9282,
  "company_financials_numeric_parses": 5542,
  "company_financials_unresolved_placeholders": 58,
  "parsing_failures": 0,
  "source_missing_cells_receiver_general": 29876,
  "hard_checks_passed": 143,
  "published_reload_passed": true,
  "row_deletions": 0,
  "imputations": 0,
  "outlier_modifications": 0,
  "FX_conversions": 0,
  "processed_files": [
    "cleaning_manifest.json",
    "cleaning_review_flags.csv",
    "cleaning_validation_report.json",
    "company_financials_standardized.csv",
    "receiver_general_standardized.csv"
  ],
  "output_hashes": {
    "cleaning_manifest.json": "720165d12ecf27e87823376621c0ae7b0b20fca91130ae4e381884e330712b51",
    "cleaning_review_flags.csv": "51a4bbe758c9fa7a014b2f563e3c1bd9c8baa182361077657e26522731ea95a5",
    "cleaning_validation_report.json": "1c9bffbf5903b27faa68a5059af42016ed01df